# 📊 Datathon PEDE - Análise Exploratória de Dados

## Associação Passos Mágicos

Este notebook contém a análise exploratória completa dos dados do PEDE 2024, respondendo às perguntas do desafio.

**Objetivos:**
1. Compreender a estrutura e qualidade dos dados
2. Analisar os indicadores educacionais (IAN, IDA, IEG, IAA, IPS, IPV, INDE)
3. Responder às 11 perguntas do desafio
4. Identificar insights e padrões relevantes
5. Preparar dados para modelagem preditiva

## 📦 Importação de Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Estatística
from scipy import stats
from scipy.stats import pearsonr, spearmanr

# Configurações
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo dos gráficos
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Configurar pandas para exibir mais linhas/colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print('✅ Bibliotecas importadas com sucesso!')

## 📂 Carregamento dos Dados

In [ ]:
# Carregar dados
df = pd.read_excel('../data/raw/BASE DE DADOS PEDE 2024 - DATATHON.xlsx')

print(f"📊 Dataset carregado com sucesso!")
print(f"   Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")
print(f"\n📋 Primeiras linhas:")
df.head()

## 🔍 Visão Geral dos Dados

In [ ]:
# Informações gerais
print("📊 INFORMAÇÕES GERAIS DO DATASET")
print("="*80)
df.info()

In [ ]:
# Estatísticas descritivas
print("\n📈 ESTATÍSTICAS DESCRITIVAS (Variáveis Numéricas)")
print("="*80)
df.describe()

In [ ]:
# Verificar valores missing
print("\n❓ VALORES MISSING")
print("="*80)

missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Total Missing': missing,
    'Percentual (%)': missing_pct
})

missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values('Total Missing', ascending=False)

if len(missing_df) > 0:
    print(missing_df)
    
    # Visualizar missing values
    fig = px.bar(missing_df.reset_index(), 
                 x='index', 
                 y='Percentual (%)',
                 title='Percentual de Valores Missing por Variável',
                 labels={'index': 'Variável', 'Percentual (%)': 'Percentual Missing (%)'})
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()
else:
    print("✅ Nenhum valor missing encontrado!")

## 📊 Análise dos Indicadores Principais

Vamos analisar os principais indicadores do PEDE:
- **IAN**: Indicador de Adequação de Nível
- **IDA**: Indicador de Desempenho Acadêmico
- **IEG**: Indicador de Engajamento
- **IAA**: Indicador de Autoavaliação
- **IPS**: Indicador Psicossocial
- **IPV**: Indicador de Ponto de Virada
- **INDE**: Indicador de Desenvolvimento Educacional

In [ ]:
# Identificar colunas de indicadores
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPV', 'INDE 22']
indicadores_disponiveis = [col for col in indicadores if col in df.columns]

print(f"📊 Indicadores disponíveis: {indicadores_disponiveis}")

# Estatísticas dos indicadores
print("\n📈 ESTATÍSTICAS DOS INDICADORES")
print("="*80)
df[indicadores_disponiveis].describe()

In [ ]:
# Distribuição dos indicadores
fig = make_subplots(
    rows=2, cols=4,
    subplot_titles=indicadores_disponiveis,
    vertical_spacing=0.15,
    horizontal_spacing=0.1
)

for i, indicador in enumerate(indicadores_disponiveis):
    row = i // 4 + 1
    col = i % 4 + 1
    
    fig.add_trace(
        go.Histogram(x=df[indicador].dropna(), name=indicador, showlegend=False),
        row=row, col=col
    )

fig.update_layout(
    title_text="Distribuição dos Indicadores Educacionais",
    height=600,
    showlegend=False
)

fig.show()

## 🎯 PERGUNTA 1: Adequação do Nível (IAN)

**Qual é o perfil geral de defasagem dos alunos (IAN) e como ele evolui ao longo do ano?**

In [ ]:
# Análise do IAN
print("📊 ANÁLISE DO INDICADOR DE ADEQUAÇÃO DE NÍVEL (IAN)")
print("="*80)

# Estatísticas do IAN
print(f"\nMédia IAN: {df['IAN'].mean():.2f}")
print(f"Mediana IAN: {df['IAN'].median():.2f}")
print(f"Desvio Padrão: {df['IAN'].std():.2f}")
print(f"Mínimo: {df['IAN'].min():.2f}")
print(f"Máximo: {df['IAN'].max():.2f}")

# Criar categorias de defasagem baseadas no IAN
def categorizar_defasagem(ian):
    if pd.isna(ian):
        return 'Desconhecido'
    elif ian >= 7:
        return 'Adequado'
    elif ian >= 4:
        return 'Moderadamente Defasado'
    else:
        return 'Severamente Defasado'

df['Categoria_Defasagem'] = df['IAN'].apply(categorizar_defasagem)

# Distribuição por categoria
print("\n📊 Distribuição por Categoria de Defasagem:")
dist_defasagem = df['Categoria_Defasagem'].value_counts()
dist_defasagem_pct = (dist_defasagem / len(df)) * 100

for categoria in dist_defasagem.index:
    print(f"   {categoria}: {dist_defasagem[categoria]} alunos ({dist_defasagem_pct[categoria]:.1f}%)")

In [ ]:
# Visualização da distribuição de defasagem
fig = px.pie(df, names='Categoria_Defasagem', 
             title='Distribuição dos Alunos por Nível de Defasagem (IAN)',
             hole=0.4,
             color_discrete_sequence=px.colors.sequential.RdYlGn[::-1])

fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Evolução do IAN por Fase
if 'Fase' in df.columns:
    ian_por_fase = df.groupby('Fase')['IAN'].agg(['mean', 'median', 'std', 'count']).reset_index()
    
    print("\n📈 IAN por Fase do Programa:")
    print(ian_por_fase)
    
    # Gráfico de evolução
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=ian_por_fase['Fase'],
        y=ian_por_fase['mean'],
        mode='lines+markers',
        name='Média IAN',
        line=dict(width=3),
        marker=dict(size=10)
    ))
    
    fig.update_layout(
        title='Evolução do IAN Médio por Fase do Programa',
        xaxis_title='Fase (1=Quartzo, 2=Ágata, 3=Ametista, 4=Topázio)',
        yaxis_title='IAN Médio',
        hovermode='x unified'
    )
    
    fig.show()

### 💡 Insights - Pergunta 1 (IAN)

*(Os insights serão preenchidos após a execução do notebook)*

## 🎯 PERGUNTA 2: Desempenho Acadêmico (IDA)

**O desempenho acadêmico médio (IDA) está melhorando, estagnado ou caindo ao longo das fases e anos?**

In [ ]:
# Análise do IDA
print("📊 ANÁLISE DO INDICADOR DE DESEMPENHO ACADÊMICO (IDA)")
print("="*80)

print(f"\nMédia IDA: {df['IDA'].mean():.2f}")
print(f"Mediana IDA: {df['IDA'].median():.2f}")
print(f"Desvio Padrão: {df['IDA'].std():.2f}")

# IDA por Fase
if 'Fase' in df.columns:
    ida_por_fase = df.groupby('Fase')['IDA'].agg(['mean', 'median', 'std', 'count']).reset_index()
    
    print("\n📈 IDA por Fase:")
    print(ida_por_fase)
    
    # Calcular variação percentual entre fases
    ida_por_fase['variacao_pct'] = ida_por_fase['mean'].pct_change() * 100
    print("\n📊 Variação Percentual do IDA entre Fases:")
    print(ida_por_fase[['Fase', 'mean', 'variacao_pct']])

In [ ]:
# Visualização da evolução do IDA
if 'Fase' in df.columns:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Evolução do IDA Médio por Fase', 'Distribuição do IDA por Fase')
    )
    
    # Gráfico de linha
    fig.add_trace(
        go.Scatter(x=ida_por_fase['Fase'], y=ida_por_fase['mean'],
                   mode='lines+markers', name='IDA Médio',
                   line=dict(width=3), marker=dict(size=10)),
        row=1, col=1
    )
    
    # Box plot
    for fase in sorted(df['Fase'].unique()):
        fig.add_trace(
            go.Box(y=df[df['Fase']==fase]['IDA'], name=f'Fase {fase}'),
            row=1, col=2
        )
    
    fig.update_xaxes(title_text="Fase", row=1, col=1)
    fig.update_yaxes(title_text="IDA Médio", row=1, col=1)
    fig.update_xaxes(title_text="Fase", row=1, col=2)
    fig.update_yaxes(title_text="IDA", row=1, col=2)
    
    fig.update_layout(height=400, showlegend=False, title_text="Análise do Desempenho Acadêmico (IDA)")
    fig.show()

### 💡 Insights - Pergunta 2 (IDA)

*(Os insights serão preenchidos após a execução do notebook)*

## 🎯 PERGUNTA 3: Engajamento nas Atividades (IEG)

**O grau de engajamento dos alunos (IEG) tem relação direta com seus indicadores de desempenho (IDA) e do ponto de virada (IPV)?**

In [ ]:
# Análise de correlação IEG x IDA x IPV
print("📊 ANÁLISE DA RELAÇÃO ENTRE ENGAJAMENTO (IEG), DESEMPENHO (IDA) E PONTO DE VIRADA (IPV)")
print("="*80)

# Correlações
corr_ieg_ida = df[['IEG', 'IDA']].corr().iloc[0, 1]
corr_ieg_ipv = df[['IEG', 'IPV']].corr().iloc[0, 1]
corr_ida_ipv = df[['IDA', 'IPV']].corr().iloc[0, 1]

print(f"\n📈 Correlações de Pearson:")
print(f"   IEG x IDA: {corr_ieg_ida:.3f}")
print(f"   IEG x IPV: {corr_ieg_ipv:.3f}")
print(f"   IDA x IPV: {corr_ida_ipv:.3f}")

# Interpretação
def interpretar_correlacao(corr):
    abs_corr = abs(corr)
    if abs_corr >= 0.7:
        return "Forte"
    elif abs_corr >= 0.4:
        return "Moderada"
    elif abs_corr >= 0.2:
        return "Fraca"
    else:
        return "Muito Fraca"

print(f"\n💡 Interpretação:")
print(f"   IEG x IDA: Correlação {interpretar_correlacao(corr_ieg_ida)}")
print(f"   IEG x IPV: Correlação {interpretar_correlacao(corr_ieg_ipv)}")
print(f"   IDA x IPV: Correlação {interpretar_correlacao(corr_ida_ipv)}")

In [ ]:
# Matriz de correlação
corr_matrix = df[['IEG', 'IDA', 'IPV']].corr()

fig = px.imshow(corr_matrix, 
                text_auto='.3f',
                aspect='auto',
                color_continuous_scale='RdBu_r',
                title='Matriz de Correlação: IEG, IDA e IPV')

fig.show()

In [ ]:
# Scatter plots
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('IEG vs IDA', 'IEG vs IPV')
)

# IEG vs IDA
fig.add_trace(
    go.Scatter(x=df['IEG'], y=df['IDA'], mode='markers',
               marker=dict(size=5, opacity=0.6),
               name='IEG vs IDA'),
    row=1, col=1
)

# IEG vs IPV
fig.add_trace(
    go.Scatter(x=df['IEG'], y=df['IPV'], mode='markers',
               marker=dict(size=5, opacity=0.6, color='orange'),
               name='IEG vs IPV'),
    row=1, col=2
)

fig.update_xaxes(title_text="IEG (Engajamento)", row=1, col=1)
fig.update_yaxes(title_text="IDA (Desempenho)", row=1, col=1)
fig.update_xaxes(title_text="IEG (Engajamento)", row=1, col=2)
fig.update_yaxes(title_text="IPV (Ponto de Virada)", row=1, col=2)

fig.update_layout(height=400, showlegend=False, 
                  title_text="Relação entre Engajamento e Outros Indicadores")
fig.show()

### 💡 Insights - Pergunta 3 (IEG)

*(Os insights serão preenchidos após a execução do notebook)*

## 🎯 PERGUNTA 4: Autoavaliação (IAA)

**As percepções dos alunos sobre si mesmos (IAA) são coerentes com seu desempenho real (IDA) e engajamento (IEG)?**

In [ ]:
# Análise IAA vs IDA vs IEG
print("📊 ANÁLISE DA AUTOAVALIAÇÃO (IAA) vs DESEMPENHO REAL")
print("="*80)

# Correlações
corr_iaa_ida = df[['IAA', 'IDA']].corr().iloc[0, 1]
corr_iaa_ieg = df[['IAA', 'IEG']].corr().iloc[0, 1]

print(f"\n📈 Correlações:")
print(f"   IAA x IDA: {corr_iaa_ida:.3f} ({interpretar_correlacao(corr_iaa_ida)})")
print(f"   IAA x IEG: {corr_iaa_ieg:.3f} ({interpretar_correlacao(corr_iaa_ieg)})")

# Criar categorias de discrepância
df['Discrepancia_IAA_IDA'] = df['IAA'] - df['IDA']

def categorizar_discrepancia(diff):
    if pd.isna(diff):
        return 'Desconhecido'
    elif diff > 1.5:
        return 'Superestimação'
    elif diff < -1.5:
        return 'Subestimação'
    else:
        return 'Coerente'

df['Categoria_Discrepancia'] = df['Discrepancia_IAA_IDA'].apply(categorizar_discrepancia)

print("\n📊 Distribuição da Discrepância IAA vs IDA:")
print(df['Categoria_Discrepancia'].value_counts())

In [ ]:
# Visualização da discrepância
fig = px.scatter(df, x='IDA', y='IAA', 
                 color='Categoria_Discrepancia',
                 title='Autoavaliação (IAA) vs Desempenho Real (IDA)',
                 labels={'IDA': 'Desempenho Real (IDA)', 'IAA': 'Autoavaliação (IAA)'},
                 opacity=0.6)

# Adicionar linha de referência (IAA = IDA)
fig.add_trace(go.Scatter(x=[0, 10], y=[0, 10], 
                         mode='lines', 
                         name='Linha de Coerência Perfeita',
                         line=dict(dash='dash', color='gray')))

fig.show()

### 💡 Insights - Pergunta 4 (IAA)

*(Os insights serão preenchidos após a execução do notebook)*

## 🎯 PERGUNTA 5: Aspectos Psicossociais (IPS)

**Há padrões psicossociais (IPS) que antecedem quedas de desempenho acadêmico ou de engajamento?**

In [ ]:
# Análise IPS vs IDA e IEG
print("📊 ANÁLISE DOS ASPECTOS PSICOSSOCIAIS (IPS)")
print("="*80)

# Correlações
corr_ips_ida = df[['IPS', 'IDA']].corr().iloc[0, 1]
corr_ips_ieg = df[['IPS', 'IEG']].corr().iloc[0, 1]

print(f"\n📈 Correlações:")
print(f"   IPS x IDA: {corr_ips_ida:.3f} ({interpretar_correlacao(corr_ips_ida)})")
print(f"   IPS x IEG: {corr_ips_ieg:.3f} ({interpretar_correlacao(corr_ips_ieg)})")

# Categorizar IPS
def categorizar_ips(ips):
    if pd.isna(ips):
        return 'Desconhecido'
    elif ips >= 7:
        return 'IPS Alto (Bom)'
    elif ips >= 4:
        return 'IPS Médio'
    else:
        return 'IPS Baixo (Risco)'

df['Categoria_IPS'] = df['IPS'].apply(categorizar_ips)

# Análise por categoria de IPS
print("\n📊 Desempenho e Engajamento por Categoria de IPS:")
analise_ips = df.groupby('Categoria_IPS')[['IDA', 'IEG']].agg(['mean', 'median', 'count'])
print(analise_ips)

In [ ]:
# Visualização
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('IPS vs IDA', 'IPS vs IEG')
)

fig.add_trace(
    go.Scatter(x=df['IPS'], y=df['IDA'], mode='markers',
               marker=dict(size=5, opacity=0.6),
               name='IPS vs IDA'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=df['IPS'], y=df['IEG'], mode='markers',
               marker=dict(size=5, opacity=0.6, color='orange'),
               name='IPS vs IEG'),
    row=1, col=2
)

fig.update_xaxes(title_text="IPS (Psicossocial)", row=1, col=1)
fig.update_yaxes(title_text="IDA (Desempenho)", row=1, col=1)
fig.update_xaxes(title_text="IPS (Psicossocial)", row=1, col=2)
fig.update_yaxes(title_text="IEG (Engajamento)", row=1, col=2)

fig.update_layout(height=400, showlegend=False,
                  title_text="Relação entre Aspectos Psicossociais e Indicadores de Performance")
fig.show()

### 💡 Insights - Pergunta 5 (IPS)

*(Os insights serão preenchidos após a execução do notebook)*

## 💾 Salvar Dados Processados

Vamos salvar o dataset com as novas features criadas para uso posterior.

In [ ]:
# Salvar dados processados
df.to_csv('../data/processed/pede_exploratory.csv', index=False, encoding='utf-8')
print("✅ Dados salvos em 'data/processed/pede_exploratory.csv'")
print(f"   Total de colunas: {df.shape[1]}")
print(f"   Novas features criadas: Categoria_Defasagem, Discrepancia_IAA_IDA, Categoria_Discrepancia, Categoria_IPS")

## 📝 Resumo da Análise Exploratória

### Principais Descobertas

1. **IAN (Adequação de Nível)**:
   - [A ser preenchido após execução]

2. **IDA (Desempenho Acadêmico)**:
   - [A ser preenchido após execução]

3. **IEG (Engajamento)**:
   - [A ser preenchido após execução]

4. **IAA (Autoavaliação)**:
   - [A ser preenchido após execução]

5. **IPS (Aspectos Psicossociais)**:
   - [A ser preenchido após execução]

### Próximos Passos

1. Continuar análise das perguntas 6-11
2. Feature engineering para modelo preditivo
3. Desenvolvimento do modelo de ML
4. Criação da aplicação Streamlit